In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path

# Style de plot
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

# Chemin vers le dataset
DATA_DIR = Path("/mnt/c/Users/enfan/Documents/StageUmons/IMS/2nd_test/")

# Vérification
files = sorted(os.listdir(DATA_DIR))
print(f"Nombre de fichiers : {len(files)}")
print(f"Premier fichier : {files[0]}")
print(f"Dernier fichier : {files[-1]}")

In [ ]:
# Charger le premier fichier
first_file = DATA_DIR / files[0]
data = np.loadtxt(first_file)

print(f"Shape : {data.shape}")
print(f"Min : {data.min():.4f}, Max : {data.max():.4f}")
print(f"Premiers échantillons :")
print(data[:5])

In [ ]:
# Constantes physiques
FS = 20000  # Fréquence d'échantillonnage : 20 kHz
N = 20480   # Nombre de points par fichier
DURATION = N / FS  # 1.024 seconde

# Vecteur temps pour l'affichage
t = np.arange(N) / FS

# Tracer les 4 canaux du premier fichier (état SAIN)
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
fig.suptitle(f"Signal vibratoire au DÉBUT du test (état sain)\nFichier : {files[0]}", fontsize=13)

bearing_names = ['Bearing 1 (futur défaut)', 'Bearing 2', 'Bearing 3', 'Bearing 4']
for i, ax in enumerate(axes):
    ax.plot(t, data[:, i], linewidth=0.5)
    ax.set_ylabel(f'{bearing_names[i]}\n(g)')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-1.5, 1.5)

axes[-1].set_xlabel('Temps (s)')
plt.tight_layout()
plt.show()

In [ ]:
# Charger le dernier fichier
last_file = DATA_DIR / files[-1]
data_end = np.loadtxt(last_file)

print(f"Shape : {data_end.shape}")
print(f"Bearing 1 - Min : {data_end[:,0].min():.3f}, Max : {data_end[:,0].max():.3f}")
print(f"Bearing 2 - Min : {data_end[:,1].min():.3f}, Max : {data_end[:,1].max():.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# DÉBUT — état sain
axes[0].plot(t, data[:, 0], linewidth=0.5, color='green')
axes[0].set_title(f'Bearing 1 - DÉBUT du test (sain)\n{files[0]}')
axes[0].set_ylabel('Accélération (g)')
axes[0].grid(True, alpha=0.3)

# FIN — état défaillant
axes[1].plot(t, data_end[:, 0], linewidth=0.5, color='red')
axes[1].set_title(f'Bearing 1 - FIN du test (défaillant)\n{files[-1]}')
axes[1].set_ylabel('Accélération (g)')
axes[1].set_xlabel('Temps (s)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def compute_fft(signal, fs):
    """Calcule la FFT en module, retourne les fréquences positives uniquement."""
    N = len(signal)
    fft_vals = np.fft.rfft(signal)
    fft_mag = np.abs(fft_vals) * 2 / N  # Normalisation
    freqs = np.fft.rfftfreq(N, d=1/fs)
    return freqs, fft_mag

# FFT du début et de la fin
freqs, fft_start = compute_fft(data[:, 0], FS)
_, fft_end = compute_fft(data_end[:, 0], FS)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(freqs, fft_start, color='green', linewidth=0.6)
axes[0].set_title(f'Spectre FFT - Bearing 1 - DÉBUT (sain)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0, 10000)  # On regarde jusqu'à 10 kHz (Nyquist = 10 kHz)

axes[1].plot(freqs, fft_end, color='red', linewidth=0.6)
axes[1].set_title(f'Spectre FFT - Bearing 1 - FIN (défaillant)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 10000)

plt.tight_layout()
plt.show()

In [ ]:
# Comparaison des amplitudes brutes
print("=== AMPLITUDES (Bearing 1) ===")
print(f"\nDÉBUT (sain):")
print(f"  RMS  = {np.sqrt(np.mean(data[:,0]**2)):.4f}")
print(f"  Min  = {data[:,0].min():.4f}")
print(f"  Max  = {data[:,0].max():.4f}")
print(f"  Peak-to-peak = {data[:,0].max() - data[:,0].min():.4f}")

print(f"\nFIN (supposément défaillant):")
print(f"  RMS  = {np.sqrt(np.mean(data_end[:,0]**2)):.4f}")
print(f"  Min  = {data_end[:,0].min():.4f}")
print(f"  Max  = {data_end[:,0].max():.4f}")
print(f"  Peak-to-peak = {data_end[:,0].max() - data_end[:,0].min():.4f}")

In [ ]:
# RMS de TOUS les canaux au début et à la fin
print("=== Comparaison de tous les canaux ===\n")
print(f"{'Canal':<12} {'Début (RMS)':<15} {'Fin (RMS)':<15} {'Ratio fin/début':<15}")
print("-" * 60)
for i in range(4):
    rms_start = np.sqrt(np.mean(data[:,i]**2))
    rms_end = np.sqrt(np.mean(data_end[:,i]**2))
    ratio = rms_end / rms_start
    print(f"Bearing {i+1:<5} {rms_start:<15.4f} {rms_end:<15.4f} {ratio:<15.2f}")

In [ ]:
# Regarder les 5 derniers fichiers et leur RMS
print("\n=== RMS des 10 derniers fichiers (Bearing 1) ===")
for f in files[-10:]:
    d = np.loadtxt(DATA_DIR / f)
    rms = np.sqrt(np.mean(d[:,0]**2))
    print(f"  {f}  →  RMS = {rms:.4f}")

In [ ]:
# Voir l'évolution sur quelques points clés
indices = [0, 100, 300, 500, 700, 800, 900, 950, 970, 983]
print("\n=== Évolution du RMS Bearing 1 aux points clés ===")
for idx in indices:
    d = np.loadtxt(DATA_DIR / files[idx])
    rms = np.sqrt(np.mean(d[:,0]**2))
    print(f"  Fichier #{idx:>4}  ({files[idx]})  →  RMS = {rms:.4f}")

In [ ]:
from tqdm.notebook import tqdm

# Calculer le RMS de tous les canaux pour tous les fichiers
print("Calcul du RMS sur les 984 fichiers (peut prendre ~3 min)...")

rms_all = np.zeros((len(files), 4))  # shape: (984, 4)
timestamps = []

for i, fname in enumerate(tqdm(files)):
    d = np.loadtxt(DATA_DIR / fname)
    rms_all[i, :] = np.sqrt(np.mean(d**2, axis=0))
    timestamps.append(fname)

print(f"\nTerminé ! Shape : {rms_all.shape}")
print(f"RMS Bearing 1 - min: {rms_all[:,0].min():.4f}, max: {rms_all[:,0].max():.4f}")

In [ ]:
# Tracer l'évolution du RMS pour les 4 roulements
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['red', 'blue', 'green', 'orange']
labels = ['Bearing 1 (défaut outer race)', 'Bearing 2', 'Bearing 3', 'Bearing 4']

for i in range(4):
    ax.plot(rms_all[:, i], color=colors[i], label=labels[i], linewidth=1, alpha=0.8)

# Marquer les zones
ax.axvspan(0, 500, alpha=0.1, color='green', label='Zone "normale" (entraînement AE)')
ax.axvspan(500, 700, alpha=0.1, color='yellow')
ax.axvspan(700, 982, alpha=0.1, color='red', label='Zone "anormale"')
ax.axvspan(982, 984, alpha=0.3, color='black', label='Machine arrêtée')

ax.set_title("Évolution du RMS sur le Set 2 (run-to-failure, ~7 jours)", fontsize=13)
ax.set_xlabel("Index du fichier (1 fichier = 10 minutes)")
ax.set_ylabel("RMS (g)")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Créer un dataset propre sans les 2 derniers fichiers
files_clean = files[:-2]
rms_clean = rms_all[:-2, :]

print(f"Dataset original : {len(files)} fichiers")
print(f"Dataset nettoyé : {len(files_clean)} fichiers")
print(f"RMS final Bearing 1 (dernier fichier valide) : {rms_clean[-1, 0]:.4f}")

In [ ]:
# Sauvegarder dans le dossier du notebook
np.save('rms_set2.npy', rms_clean)
print("RMS sauvegardé dans rms_set2.npy")

# Sauvegarder aussi la liste des fichiers
with open('files_set2.txt', 'w') as f:
    for fname in files_clean:
        f.write(fname + '\n')
print("Liste des fichiers sauvegardée dans files_set2.txt")

In [ ]:
# Tracer l'évolution du RMS pour les 4 roulements (sans les 2 derniers points)
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['red', 'blue', 'green', 'orange']
labels = ['Bearing 1 (défaut outer race)', 'Bearing 2', 'Bearing 3', 'Bearing 4']

# rms_clean a déjà les 2 derniers points exclus
for i in range(4):
    ax.plot(rms_clean[:, i], color=colors[i], label=labels[i], linewidth=1, alpha=0.8)

# Marquer les zones de labellisation
ax.axvspan(0, 500, alpha=0.1, color='green', label='Zone "normale" (entraînement AE)')
ax.axvspan(500, 700, alpha=0.1, color='yellow', label='Zone grise')
ax.axvspan(700, len(rms_clean), alpha=0.1, color='red', label='Zone "anormale"')

ax.set_title("Évolution du RMS sur le Set 2 (run-to-failure, ~7 jours)", fontsize=13)
ax.set_xlabel("Index du fichier (1 fichier = 10 minutes)")
ax.set_ylabel("RMS (g)")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Vérification de l'état des variables
print(f"Shape rms_all : {rms_all.shape}")
print(f"Dernières valeurs de rms_all (Bearing 1) :")
print(rms_all[-5:, 0])

In [ ]:
# Recréer le dataset propre en excluant les 2 derniers points
files_clean = files[:-2]          # 982 fichiers au lieu de 984
rms_clean = rms_all[:-2, :].copy()  # On copie pour être sûr

# Vérification
print(f"files_clean : {len(files_clean)} fichiers (devrait être 982)")
print(f"rms_clean shape : {rms_clean.shape} (devrait être (982, 4))")
print(f"Dernières valeurs de rms_clean (Bearing 1) :")
print(rms_clean[-5:, 0])

In [ ]:
# Tracer l'évolution du RMS - VERSION PROPRE
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['red', 'blue', 'green', 'orange']
labels = ['Bearing 1 (défaut outer race)', 'Bearing 2', 'Bearing 3', 'Bearing 4']

for i in range(4):
    ax.plot(rms_clean[:, i], color=colors[i], label=labels[i], linewidth=1, alpha=0.8)

# Zones de labellisation
ax.axvspan(0, 500, alpha=0.1, color='green', label='Zone "normale" (entraînement AE)')
ax.axvspan(500, 700, alpha=0.1, color='yellow', label='Zone grise')
ax.axvspan(700, len(rms_clean), alpha=0.1, color='red', label='Zone "anormale"')

ax.set_title("Évolution du RMS sur le Set 2 (run-to-failure, ~7 jours)", fontsize=13)
ax.set_xlabel("Index du fichier (1 fichier = 10 minutes)")
ax.set_ylabel("RMS (g)")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print("Shape rms_all :", rms_all.shape)
print("Shape rms_clean :", rms_clean.shape)
print()
print("10 dernières valeurs de rms_clean (Bearing 1, colonne 0) :")
print(rms_clean[-10:, 0])
print()
print("Max de rms_clean (Bearing 1) :", rms_clean[:, 0].max())
print("Min de rms_clean (Bearing 1) :", rms_clean[:, 0].min())
print("Argmax (index du pic) :", rms_clean[:, 0].argmax())
print("Argmin (index du minimum) :", rms_clean[:, 0].argmin())

In [ ]:
# Zoom sur les 30 derniers fichiers de rms_clean
fig, ax = plt.subplots(figsize=(12, 5))
last_30_idx = np.arange(len(rms_clean) - 30, len(rms_clean))
ax.plot(last_30_idx, rms_clean[-30:, 0], 'ro-', markersize=5, linewidth=1.5)
ax.set_title("Zoom : 30 derniers fichiers de rms_clean (Bearing 1)")
ax.set_xlabel("Index du fichier")
ax.set_ylabel("RMS (g)")
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Comparer rms_all (avec les 2 derniers à zéro) vs rms_clean (sans)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Avec les 2 derniers points à zéro
axes[0].plot(rms_all[:, 0], 'r-', linewidth=1)
axes[0].set_title("rms_all (984 fichiers) - avec la chute 'machine éteinte' à la fin")
axes[0].set_ylabel("RMS (g)")
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=982, color='black', linestyle='--', alpha=0.5, label='Index 982')
axes[0].legend()

# Sans les 2 derniers points
axes[1].plot(rms_clean[:, 0], 'r-', linewidth=1)
axes[1].set_title("rms_clean (982 fichiers) - sans la chute artificielle")
axes[1].set_xlabel("Index du fichier")
axes[1].set_ylabel("RMS (g)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Choisir 4 fichiers représentatifs des différents stades
stages = {
    'Sain (début)':      100,   # Régime normal stable
    'Sain tardif':       450,   # Toujours stable
    'Dégradation modérée': 800, # Le défaut s'installe
    'Défaillance avancée': 970, # Juste avant la failure
}

# Charger les 4 fichiers et calculer leurs FFT
fft_data = {}
for stage_name, idx in stages.items():
    signal = np.loadtxt(DATA_DIR / files_clean[idx])[:, 0]  # Bearing 1
    freqs, fft_mag = compute_fft(signal, FS)
    fft_data[stage_name] = (freqs, fft_mag, idx)

# Tracer les 4 spectres
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
colors = ['green', 'darkgreen', 'orange', 'red']

for ax, (stage_name, (freqs, fft_mag, idx)), color in zip(axes, fft_data.items(), colors):
    ax.plot(freqs, fft_mag, color=color, linewidth=0.6)
    rms_val = rms_clean[idx, 0]
    ax.set_title(f"{stage_name} — Fichier #{idx} — RMS = {rms_val:.3f} g")
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1000)  # On regarde la bande 0-1 kHz (où vivent les défauts de roulement)

axes[-1].set_xlabel("Fréquence (Hz)")
plt.tight_layout()
plt.show()

In [ ]:
# Fréquences théoriques de défaut pour roulement ZA-2115 à 2000 RPM
defect_freqs = {
    'FTF (cage)': 14.99,
    'BPFO (bague ext.)': 236.40,
    'BSF (rouleau)': 277.95,
    'BPFI (bague int.)': 297.94,
}

fig, ax = plt.subplots(figsize=(14, 6))

# Superposer les 4 spectres dans la zone d'intérêt
for (stage_name, (freqs, fft_mag, idx)), color in zip(fft_data.items(), colors):
    ax.plot(freqs, fft_mag, color=color, linewidth=0.8, alpha=0.8, label=f"{stage_name} (#{idx})")

# Marquer les fréquences caractéristiques
for name, freq in defect_freqs.items():
    ax.axvline(x=freq, linestyle='--', color='black', alpha=0.4)
    ax.text(freq, ax.get_ylim()[1]*0.95, name, rotation=90, fontsize=9, 
            verticalalignment='top', alpha=0.7)

ax.set_title("Évolution du spectre FFT - Bearing 1 (zoom 0-500 Hz)")
ax.set_xlabel("Fréquence (Hz)")
ax.set_ylabel("Amplitude")
ax.set_xlim(0, 500)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks

# Spectre stade défaillance
signal_def = np.loadtxt(DATA_DIR / files_clean[970])[:, 0]
freqs_def, fft_def = compute_fft(signal_def, FS)

# Zoomer sur la zone 800-1100 Hz
mask = (freqs_def >= 800) & (freqs_def <= 1100)
freqs_zoom = freqs_def[mask]
fft_zoom = fft_def[mask]

# Détecter les pics
threshold = fft_zoom.max() * 0.2
peaks, _ = find_peaks(fft_zoom, height=threshold, distance=10)

print("=== Pics dans la bande 800-1100 Hz (défaillance) ===")
for p in peaks:
    print(f"  Fréquence : {freqs_zoom[p]:.1f} Hz, Amplitude : {fft_zoom[p]:.5f}")

# Comparer avec stade sain
signal_sain = np.loadtxt(DATA_DIR / files_clean[100])[:, 0]
freqs_s, fft_s = compute_fft(signal_sain, FS)
fft_zoom_s = fft_s[mask]
peaks_s, _ = find_peaks(fft_zoom_s, height=fft_zoom_s.max()*0.2, distance=10)

print("\n=== Pics dans la bande 800-1100 Hz (sain) ===")
for p in peaks_s:
    print(f"  Fréquence : {freqs_zoom[p]:.1f} Hz, Amplitude : {fft_zoom_s[p]:.5f}")

In [ ]:
def energy_in_band(signal, fs, f_low, f_high):
    """Calcule l'énergie spectrale dans une bande [f_low, f_high]."""
    freqs, fft_mag = compute_fft(signal, fs)
    mask = (freqs >= f_low) & (freqs <= f_high)
    return np.sum(fft_mag[mask] ** 2)

# Bandes ajustées avec les fréquences réelles observées
# BPFO réel ≈ 230 Hz (vitesse 1950 RPM)
bands = {
    'Basse (0-100)':          (0, 100),       # Rotation + cage
    'BPFO fond. (200-260)':   (200, 260),     # 1×BPFO ≈ 230 Hz
    '2×BPFO (440-500)':       (440, 500),     # 2×BPFO ≈ 461 Hz
    '3×BPFO (670-730)':       (670, 730),     # 3×BPFO ≈ 692 Hz
    '4×BPFO (900-950)':       (900, 950),     # 4×BPFO ≈ 923 Hz
    'Résonance (970-1010)':   (970, 1010),    # Résonance structurelle ~985 Hz
    'HF (1100-3000)':         (1100, 3000),   # Bruit large bande
}

# Tableau aux 4 stades
print(f"{'Bande':<28}", end="")
for stage in stages.keys():
    print(f"{stage[:18]:<22}", end="")
print()
print("-" * 130)

# On stocke aussi les valeurs pour calculer les ratios
energies = {stage: {} for stage in stages.keys()}

for band_name, (f_low, f_high) in bands.items():
    print(f"{band_name:<28}", end="")
    for stage_name, idx in stages.items():
        signal = np.loadtxt(DATA_DIR / files_clean[idx])[:, 0]
        energy = energy_in_band(signal, FS, f_low, f_high)
        energies[stage_name][band_name] = energy
        print(f"{energy:<22.6f}", end="")
    print()

# Calcul des ratios défaillance/sain
print("\n" + "=" * 130)
print("RATIOS défaillance / sain (>10 = très discriminant)")
print("=" * 130)
print(f"{'Bande':<28} {'Ratio':<15} {'Évaluation'}")
print("-" * 130)
for band_name in bands.keys():
    e_sain = energies['Sain (début)'][band_name]
    e_def = energies['Défaillance avancée'][band_name]
    ratio = e_def / max(e_sain, 1e-12)  # Éviter div par zéro
    if ratio > 20:
        eval_str = "🔥🔥🔥 EXCELLENTE feature"
    elif ratio > 5:
        eval_str = "✅ Bonne feature"
    elif ratio > 2:
        eval_str = "⚠️ Feature modérée"
    else:
        eval_str = "❌ Peu informatif"
    print(f"{band_name:<28} {ratio:<15.1f} {eval_str}")

In [ ]:
# Bar chart comparatif des énergies par bande
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(bands))
width = 0.2

colors = ['lightgreen', 'green', 'orange', 'red']
for i, (stage_name, idx) in enumerate(stages.items()):
    values = [energies[stage_name][band] for band in bands.keys()]
    ax.bar(x + i*width, values, width, label=stage_name, color=colors[i])

ax.set_yscale('log')  # Échelle log car les amplitudes varient beaucoup
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels(bands.keys(), rotation=30, ha='right')
ax.set_ylabel("Énergie (échelle log)")
ax.set_title("Énergie spectrale par bande aux 4 stades de dégradation")
ax.legend()
ax.grid(True, alpha=0.3, axis='y', which='both')

plt.tight_layout()
plt.show()

In [ ]:
# Comparer les spectres aux 4 stades sur toute la bande (0-10 kHz Nyquist)
fig, ax = plt.subplots(figsize=(14, 6))

for (stage_name, idx), color in zip(stages.items(), ['green', 'darkgreen', 'orange', 'red']):
    signal = np.loadtxt(DATA_DIR / files_clean[idx])[:, 0]
    freqs, fft_mag = compute_fft(signal, FS)
    ax.semilogy(freqs, fft_mag + 1e-8, color=color, linewidth=0.5, alpha=0.7, 
                label=f"{stage_name} (#{idx})")

ax.set_title("Spectres complets en échelle log - Bearing 1")
ax.set_xlabel("Fréquence (Hz)")
ax.set_ylabel("Amplitude (log)")
ax.set_xlim(0, 10000)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
# Énergie globale dans la bande HF (1100-10000 Hz) à chaque stade
print("=== Énergie HF (1100-10000 Hz) aux 4 stades ===\n")
hf_energies = []
for stage_name, idx in stages.items():
    signal = np.loadtxt(DATA_DIR / files_clean[idx])[:, 0]
    e_hf = energy_in_band(signal, FS, 1100, 10000)
    hf_energies.append(e_hf)
    print(f"  {stage_name:<25} #{idx:<4} → Énergie HF = {e_hf:.5f}")

# Ratio
ratio_hf = hf_energies[-1] / max(hf_energies[0], 1e-12)
print(f"\nRatio Défaillance/Sain en HF : {ratio_hf:.1f}")
if ratio_hf > 5:
    print("→ ✅ La bande HF mérite d'être incluse comme feature")
elif ratio_hf > 2:
    print("→ ⚠️ La bande HF apporte une info modérée")
else:
    print("→ ❌ La bande HF n'est pas discriminante, à exclure")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True, sharey=True)

colors = ['green', 'darkgreen', 'orange', 'red']

for ax, (stage_name, idx), color in zip(axes, stages.items(), colors):
    signal = np.loadtxt(DATA_DIR / files_clean[idx])[:, 0]
    freqs, fft_mag = compute_fft(signal, FS)
    ax.semilogy(freqs, fft_mag + 1e-8, color=color, linewidth=0.5)
    ax.set_title(f"{stage_name} (#{idx})", fontsize=11)
    ax.set_ylabel("Amplitude (log)")
    ax.set_xlim(0, 10000)
    ax.grid(True, alpha=0.3, which='both')
    ax.set_ylim(1e-6, 1e-1)  # Même échelle pour comparer

axes[-1].set_xlabel("Fréquence (Hz)")
plt.suptitle("Spectres complets en échelle log - Bearing 1 (4 stades)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import uniform_filter1d

fig, ax = plt.subplots(figsize=(14, 6))

for (stage_name, idx), color in zip(stages.items(), colors):
    signal = np.loadtxt(DATA_DIR / files_clean[idx])[:, 0]
    freqs, fft_mag = compute_fft(signal, FS)
    # Lissage sur ~20 points (≈ 20 Hz de résolution)
    fft_smooth = uniform_filter1d(fft_mag, size=20)
    ax.semilogy(freqs, fft_smooth + 1e-8, color=color, linewidth=1.2, 
                alpha=0.8, label=f"{stage_name} (#{idx})")

ax.set_title("Spectres lissés (moyenne glissante 20 Hz) - Bearing 1")
ax.set_xlabel("Fréquence (Hz)")
ax.set_ylabel("Amplitude lissée (log)")
ax.set_xlim(0, 10000)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Tes données
stage_names = ['Sain\n(début)\n#100', 'Sain\ntardif\n#450', 'Dégradation\nmodérée\n#800', 'Défaillance\navancée\n#970']
hf_energies = [0.00853, 0.00807, 0.01810, 0.21935]
colors = ['#2ecc71', '#27ae60', '#f39c12', '#e74c3c']  # Vert clair → Vert → Orange → Rouge

fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.bar(stage_names, hf_energies, color=colors, edgecolor='black', linewidth=1.2)

# Ajouter les valeurs au-dessus des barres
for bar, val in zip(bars, hf_energies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.5f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Mise en forme
ax.set_title("Énergie spectrale dans la bande HF (1100-10000 Hz)\nBearing 1 - Set 2", 
             fontsize=14, fontweight='bold', pad=20)
ax.set_ylabel("Énergie spectrale", fontsize=12)
ax.set_ylim(0, max(hf_energies) * 1.15)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')
ax.set_axisbelow(True)

# Annotation du ratio frappant
ax.annotate('', xy=(3, 0.21935), xytext=(0, 0.00853),
            arrowprops=dict(arrowstyle='->', color='red', lw=2, alpha=0.7))
ax.text(1.5, 0.13, '×25.7', fontsize=20, color='red', fontweight='bold',
        ha='center', bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='red', linewidth=2))

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

bars = ax.bar(stage_names, hf_energies, color=colors, edgecolor='black', linewidth=1.2)

# Valeurs au-dessus
for bar, val in zip(bars, hf_energies):
    ax.text(bar.get_x() + bar.get_width()/2., val * 1.1,
            f'{val:.5f}', ha='center', va='bottom', fontsize=10)

ax.set_yscale('log')
ax.set_title("Énergie spectrale HF (1100-10000 Hz) - échelle logarithmique\nBearing 1 - Set 2", 
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Énergie spectrale (log)", fontsize=12)
ax.grid(True, alpha=0.3, axis='y', which='both', linestyle='--')
ax.set_axisbelow(True)
ax.set_ylim(0.005, 0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Données
stage_names = ['Sain (début)\n#100', 'Sain tardif\n#450', 'Dégradation\nmodérée\n#800', 'Défaillance\navancée\n#970']
hf_energies = [0.00853, 0.00807, 0.01810, 0.21935]
x = np.arange(len(stage_names))

# Figure
fig, ax = plt.subplots(figsize=(11, 6))

# Courbe principale
ax.plot(x, hf_energies, 'o-', color='#c0392b', linewidth=2.5, 
        markersize=11, markerfacecolor='#e74c3c', markeredgewidth=2, 
        markeredgecolor='darkred', label='Énergie HF (1100-10000 Hz)')

# Valeurs au-dessus de chaque point
for xi, yi in zip(x, hf_energies):
    ax.annotate(f'{yi:.5f}', xy=(xi, yi), xytext=(0, 12),
                textcoords='offset points', ha='center',
                fontsize=11, fontweight='bold', color='#c0392b')

# Mise en forme
ax.set_xticks(x)
ax.set_xticklabels(stage_names, fontsize=11)
ax.set_title("Évolution de l'énergie spectrale HF (1100-10000 Hz)\nBearing 1 - Set 2 IMS", 
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Énergie spectrale", fontsize=12)
ax.set_ylim(0, max(hf_energies) * 1.18)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

ax.semilogy(x, hf_energies, 'o-', color='#c0392b', linewidth=2.5,
            markersize=11, markerfacecolor='#e74c3c', markeredgewidth=2,
            markeredgecolor='darkred', label='Énergie HF (1100-10000 Hz)')

for xi, yi in zip(x, hf_energies):
    ax.annotate(f'{yi:.5f}', xy=(xi, yi), xytext=(10, 0),
                textcoords='offset points', va='center',
                fontsize=11, fontweight='bold', color='#c0392b')

ax.set_xticks(x)
ax.set_xticklabels(stage_names, fontsize=11)
ax.set_title("Évolution de l'énergie spectrale HF - échelle logarithmique\nBearing 1 - Set 2 IMS", 
             fontsize=14, fontweight='bold', pad=15)
ax.set_ylabel("Énergie spectrale (log)", fontsize=12)
ax.grid(True, alpha=0.3, linestyle='--', which='both')
ax.set_axisbelow(True)
ax.legend(loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
from tqdm.notebook import tqdm

print("Calcul de l'énergie HF pour les 982 fichiers (~3-5 min)...")

hf_all = np.zeros(len(files_clean))

for i, fname in enumerate(tqdm(files_clean)):
    signal = np.loadtxt(DATA_DIR / fname)[:, 0]  # Bearing 1
    hf_all[i] = energy_in_band(signal, FS, 1100, 10000)

print(f"\nTerminé !")
print(f"Min : {hf_all.min():.5f}")
print(f"Max : {hf_all.max():.5f}")
print(f"Ratio max/min : {hf_all.max()/hf_all.min():.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Courbe principale
ax.plot(hf_all, color='#c0392b', linewidth=1, alpha=0.9)

# Zones de labellisation
ax.axvspan(0, 500, alpha=0.1, color='green', label='Zone "normale"')
ax.axvspan(500, 700, alpha=0.1, color='yellow', label='Zone grise')
ax.axvspan(700, len(hf_all), alpha=0.1, color='red', label='Zone "anormale"')

# Marquer les 4 stades qu'on a analysés
stage_indices = [100, 450, 800, 970]
stage_labels = ['Sain (début)', 'Sain tardif', 'Dégradation', 'Défaillance']
stage_colors = ['#2ecc71', '#27ae60', '#f39c12', '#c0392b']

for idx, lbl, col in zip(stage_indices, stage_labels, stage_colors):
    ax.plot(idx, hf_all[idx], 'o', color=col, markersize=12, 
            markeredgecolor='black', markeredgewidth=1.5, zorder=5)
    ax.annotate(f'{lbl}\n(#{idx})', xy=(idx, hf_all[idx]), 
                xytext=(0, 15), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color=col,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                         edgecolor=col, alpha=0.9))

ax.set_title("Évolution de l'énergie spectrale HF (1100-10000 Hz)\nBearing 1 - Set 2 IMS (982 fichiers)", 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Index du fichier (1 fichier = 10 minutes)", fontsize=12)
ax.set_ylabel("Énergie spectrale HF", fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

# Courbe en log
ax.semilogy(hf_all, color='#c0392b', linewidth=1, alpha=0.9)

# Zones de labellisation
ax.axvspan(0, 500, alpha=0.1, color='green', label='Zone "normale"')
ax.axvspan(500, 700, alpha=0.1, color='yellow', label='Zone grise')
ax.axvspan(700, len(hf_all), alpha=0.1, color='red', label='Zone "anormale"')

# Marquer les 4 stades
for idx, lbl, col in zip(stage_indices, stage_labels, stage_colors):
    ax.plot(idx, hf_all[idx], 'o', color=col, markersize=12, 
            markeredgecolor='black', markeredgewidth=1.5, zorder=5)
    ax.annotate(f'{lbl}\n(#{idx})', xy=(idx, hf_all[idx]), 
                xytext=(0, 20), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color=col,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', 
                         edgecolor=col, alpha=0.9))

ax.set_title("Évolution de l'énergie HF en échelle logarithmique\nBearing 1 - Set 2 IMS", 
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel("Index du fichier (1 fichier = 10 minutes)", fontsize=12)
ax.set_ylabel("Énergie spectrale HF (log)", fontsize=12)
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--', which='both')
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

In [ ]:
# Sauvegarder l'énergie HF de tous les fichiers
np.save('hf_energy_set2.npy', hf_all)
print("Énergie HF sauvegardée dans hf_energy_set2.npy")